# 🛟 Support bounce — buy a tested support level, trail the stop, avoid the drawdown

**The idea.** A horizontal level that recently stopped the selling tends to stop it again: published support/resistance levels predicted intraday trend interruptions far better than chance (Osler, FRBNY 2000), and such levels sit exactly where resting bid depth peaks in the limit order book (Kavajecz & Odders-White 2004). The strategy buys *near* a recently tested support — the lowest low of the last `level_window` minutes, tested at least `min_touches` times — in a non-bear regime, with everything sized in ATRs: the touch band (`touch_tol`), the entry zone (`entry_dist`), and a stop deliberately **below the level's noise band** (`sl_atr`; `valid()` refuses tight stops, because Kaminski & Lo showed a tight stop under a mean-reversion entry systematically sells the bounce). Full thesis and literature in [README.md](./README.md).

**The capital and the yardstick.** The whole strategy trades **ONE shared $10k wallet** (`cash_frac`: each entry deploys 25% of the free cash, so total invested can never exceed the book — a real single account, not one account per symbol). Everything is judged against an equal-weight buy-and-hold of the same names with the same $10k: the optimizer maximizes **`info_ratio` and `excess_cagr`**, both benchmark-relative and annualized, so a score above zero means *beating the market*, not just being long in a rising one.

**The experiment.** Three contentious choices are searched first-class, so the data — not taste — arbitrates: **`entry_mode`** (`touch` = enter on the dip itself, the closest no-look-ahead stand-in for the resting limit order of the original idea; `bounce` = demand the close back above the band first, the adverse-selection-aware variant), **`sl_trail`** (trail the stop behind the best price to lock in profit, vs a fixed stop), and **`tf_minutes`** (the dip trigger evaluated on 1/5/15-minute windows derived from the same 1m bars — "which candle interval makes most sense" without any extra data). `min_touches: 0` is the ablation control: if it dominates the front, the "tested level" story added nothing over generic dip-buying — that negative result is a deliverable, not a failure.

**The universe.** `HighOctaneTop20`: the sibling repo's **HighOctane** universe (top volatility tercile within sector), narrowed to the **20 most liquid names by trailing-90-day average dollar volume** (`tsim subset --universe HighOctane --top 20`) — volatile enough to test levels intraday, liquid enough for honest next-bar-open fills.

**Train vs holdout.** The `data:` block sets `end` to a deliberate **train cutoff** — the sweep and the optimizer only ever see data up to it. §11 replays the picked parameters on the untouched ~2-month tail against the hurdle (holdout `excess_cagr` and `info_ratio` both positive), §12 re-prices them under stressed costs, and §13 slices everything by `tf_minutes` to answer the interval question. Every knob lives in [`config.yaml`](./config.yaml); this notebook only *drives* the shared engine: **load → overview → backtest → inspect → sweep → optimize → boundary-check → verify → out-of-sample → cost stress → interval read-out**.

## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import backtest as bt, jobs, optimize as opt, strategies
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple import plots

strat = strategies.load("support_bounce")
print("data :", strat.data)
print("params:", strat.params)

In [ ]:
d = strat.data
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]),
                         d["bar"], d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
print(f"panel: {panel.close.shape[0]:,} bars x {len(panel.symbols)} symbols "
      f"({panel.close.index[0]:%Y-%m-%d} → {panel.close.index[-1]:%Y-%m-%d})")

## 2 · Strategy overview — hyperparameters and search ranges

Every window parameter is expressed in **minutes** and converted to bars inside
`signals()` from `panel.bar` (on this 1m config, minutes == bars) — that is what
lets `tf_minutes` compare candle intervals on a single dataset, and the same
config run on 5m/1h/1d panels unchanged.

| param | role in the strategy | intuition |
|---|---|---|
| `level_window` | support lookback (minutes) | the level's lifetime: rolling min of the low, shifted |
| `min_touches` | prior distinct tests required | **the ablation**: 0 = untested levels trade too |
| `touch_tol` | touch band in ATRs above the level | what "testing the level" means — and the dip target |
| `entry_dist` | entry zone in ATRs above the level | the selectivity knob: the close must be near the level |
| `entry_mode` | touch / bounce | **the experiment**: anticipatory dip vs confirmed reclaim |
| `tf_minutes` | trigger dip window (minutes) | **the interval question**: 1/5/15m candles from 1m bars |
| `ema_fast` / `ema_slow` / `regime_dz` | regime EMAs + dead zone | bull / bear / sideways; bear never enters |
| `allow_sideways` | sideways entries allowed? | bull-only vs bull+sideways |
| `atr_window` | ATR smoothing (minutes) | how fast bands + stops adapt to a name's vol |
| `sl_atr` | stop distance in ATRs | wide by design; `valid()` forces it past `entry_dist + touch_tol` |
| `sl_trail` | trail the stop? | **the user question**: lock in profit vs give the bounce room |
| `tp_mult` | target in ATRs | 0 = run to the time/regime/EOD exits |
| `max_hold` | time stop (minutes held) | bounces that never come get cut |
| `exit_on_bear` | exit on a regime flip? | drop the bet when its justification dies |
| `eod_exit` | flat at session close? | `false` lets the overnight gap carry the reversion |
| `skip_open` / `skip_close` | entry blackout (minutes) | no fills into open/close chaos; inert on 1d |
| `cash_frac` | fraction of free cash per entry | ONE shared $10k wallet — never more than 100% invested |

The table below shows, per parameter: the **default** used by the single
backtest (§3), the hand-picked **sweep grid** (§5), and the **Optuna search
range** (§6). A `—` means the stage keeps the default.

In [ ]:
import pandas as pd

def _fmt_space(spec):
    """One readable line for an optuna.space entry."""
    if spec["type"] == "categorical":
        return " | ".join(map(str, spec["choices"]))
    tail = " (log)" if spec.get("log") else (f", step {spec['step']}" if spec.get("step") else "")
    return f"{spec['low']} – {spec['high']}{tail}"

pd.DataFrame({
    "default": pd.Series(strat.params),
    "sweep grid (§5)": pd.Series({k: ", ".join(map(str, v)) for k, v in strat.sweep.items()}),
    "optuna search range (§6)": pd.Series({k: _fmt_space(s)
                                           for k, s in strat.optuna["space"].items()}),
}).fillna("—")

## 3 · Single backtest with the config defaults

Engine rules (identical for every strategy): signals execute at the **next
bar's adjusted open** and realistic costs are always charged. This strategy
runs the **cash-shared wallet**: ONE $10k book for the whole universe, each
entry deploying `cash_frac` of the free cash — total invested can never exceed
the wallet, like a real single account. The stats table shows the absolutes
plus the numbers that actually matter: `cagr` (annualized return) and the
benchmark-relative `excess_cagr` / `info_ratio` vs an equal-weight buy-and-hold
of the same panel — the pair the optimizer maximizes.

In [ ]:
import pandas as pd
pf = bt.run_strategy(strat, panel, settings=settings)
stats = bt.portfolio_stats(pf)
fees, slippage = bt.leg_costs(settings.costs)
stats |= bt.active_stats(pf, panel, fees=fees, slippage=slippage,
                         total_return=stats["total_return"])
pd.Series(stats).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity, config defaults").show()

## 4 · Inspect a symbol — trades and level overlays

`inspect_symbol` is the microscope of this notebook, used twice: here on the
most-traded symbol with the *default* parameters, and again in §10 on the
*optimized* ones. How to read the figure:

* **Price panel** — the support line (rolling min of the last `level_window`
  minutes, drawn from close as a proxy; signals use the true low) with its
  `touch_tol`-ATR **touch band** and `entry_dist`-ATR **zone top**, plus the
  two regime EMAs. ▲ blue = long entry: the trailing `tf_minutes` low reached
  the touch band while the close held inside the zone in a non-bear regime —
  in `bounce` mode the close had also reclaimed the band. ✕ = exit: the
  `sl_atr` ATR stop (trailing when `sl_trail`), the optional `tp_mult` target,
  the `max_hold` time stop, a bear regime flip, or (with `eod_exit`) the
  session close. The entry→exit segment is green when the trade won, red when
  it lost.
* **Momentum panel** — fast EMA vs slow EMA: the regime gate (positive = bull,
  around zero = sideways, negative = bear, where entries are vetoed).
* **Volume panel** — unused by this strategy.

In [ ]:
def inspect_symbol(sym, pf, params):
    """Deep-dive one symbol of a portfolio: annotated chart + its trade table."""
    all_trades = bt.trade_records(pf)
    merged = {**strat.params, **params, "bar": panel.bar, "tz": panel.tz}
    overlays = (strat.indicators(panel.close[sym], panel.volume[sym], **merged)
                if strat.indicators else None)
    plots.plot_symbol(panel.close[sym], panel.volume[sym], all_trades, symbol=sym,
                      overlays=overlays,
                      equity=pf.value(group_by=False)[sym]).show()  # symbol slice of the wallet
    mine = all_trades[all_trades["symbol"] == sym]
    return mine.sort_values("entry_ts").reset_index(drop=True).round(3)

trades = bt.trade_records(pf)
busiest = trades["symbol"].value_counts().index[0]
inspect_symbol(busiest, pf, strat.params)

## 5 · Grid sweep — a coarse map of the parameter space

Before optimizing, sweep the small hand-picked grid from `config.yaml`. The
ranked bars show the best single combos; the **heatmap grid** below them is the
multi-objective view — the same parameter pair judged by `excess_cagr`
(annualized edge over buy-and-hold), `cagr` *and* breadth (`pct_profitable` =
share of traded names that made money) at once, so a region that pays on
return but is carried by one name stands out immediately. (The sweep reports
`excess_cagr` but not `info_ratio` — that needs per-combo equity curves the
column-stacked sweep does not keep; §6's optimizer evaluates both.)

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
print(f"{len(grid)} combos")
sweep_stats = bt.sweep(strat, panel, grid, settings=settings)
sweep_stats.sort_values("excess_cagr", ascending=False).head(8).round(3)

In [ ]:
plots.plot_sweep(sweep_stats).show()
plots.plot_heatmap_grid(
    sweep_stats,
    pairs=[("tf_minutes", "entry_mode"), ("tf_minutes", "sl_trail")],
    metrics=("excess_cagr", "cagr", "pct_profitable"),
).show()

## 6 · Optimize (resumable, runs outside the notebook)

`jobs.run_optimize` hands the work to a `tsim optimize` subprocess. The study
maximizes **`info_ratio` and `excess_cagr` together** — both measured against
an equal-weight buy-and-hold of the same panel, so a trial scores above zero
only by *beating the market*, not by being long in a rising one; both
annualized, so windows of different lengths stay comparable. The result is a
*Pareto front* — the set of parameter sets where improving one objective must
hurt the other. Every finished trial is checkpointed to `studies/{study}.log`;
**re-running this cell resumes** exactly where it stopped (Ctrl-C / kernel
restarts lose nothing).

While it runs, every finished trial streams one line below — its objectives,
the current front size, and its parameters — so you can watch the search move.
For a richer live view, copy the **`uvx optuna-dashboard …`** command from the
first output line into a second terminal: it serves parameter importances,
objective history and the front at http://127.0.0.1:8080 while the study runs.

In [ ]:
N_TRIALS = 1500   # raise for a deeper search; re-runs accumulate into the same study
jobs.run_optimize("support_bounce", n_trials=N_TRIALS)

## 7 · Read the study: Pareto front and the picked trial

Gray dots are all completed trials; the blue **staircase** is the exact
boundary of the dominated region (each labeled point is one Pareto-optimal
trial); the orange star is the trial `pick_best` selects under the weights
below. Hover any point for its full parameter set.

One strategy-specific caveat: the entry is an AND-stack of gates
(zone ∧ dip ∧ regime ∧ tested ∧ reclaim) — a selectivity dial. If the
admissible set below comes up empty, the space skewed too selective for this
tape; widen `entry_dist`/`touch_tol` in `config.yaml` and resume §6 before
believing any pick.

In [ ]:
study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)

# Objectives are benchmark-relative and annualized: info_ratio (active return per
# unit of tracking risk) and excess_cagr (annualized edge over equal-weight
# buy-and-hold). Two guards before picking: (1) an admissibility floor on train
# trades — a handful of lucky fills can post a huge info_ratio that means nothing;
# (2) weights leaning on excess_cagr, the economically-sized objective, with
# info_ratio as the tiebreaker. Sanity-check the pick in §10 and §11.
MIN_TRADES = 300      # admissibility floor for TRAIN trades — below this, ranking is noise
WEIGHTS = (0.5, 1.0)  # (info_ratio, excess_cagr): lean on the annualized edge itself

eligible = front[front["n_trades"] >= MIN_TRADES]
if eligible.empty:
    print(f"!! no Pareto trial reaches {MIN_TRADES} train trades — showing the raw front; "
          "treat any pick as inadmissible")
    eligible = front
objs = list(opt.OBJECTIVES)
span = (eligible[objs].max() - eligible[objs].min()).replace(0, 1.0)
score = ((eligible[objs] - eligible[objs].min()) / span).mul(
    pd.Series(dict(zip(objs, WEIGHTS, strict=True)))).sum(axis=1)
best = next(t for t in study.best_trials
            if t.number == int(eligible.loc[score.idxmax(), "trial"]))
print(f"front: {len(front)} trials, {len(eligible)} admissible (>= {MIN_TRADES} trades); "
      f"picked trial {best.number}")
plots.plot_pareto(front, everything, picked=best.number).show()
eligible.head(8).round(3)

## 8 · Boundary check — did the optimum hit the search-space edge?

A picked trial sitting **on the edge of a searched range** is a hint that the
true optimum may lie outside it. `optimize.boundary_report` compares the
picked parameters (and the whole Pareto front) with `optuna.space` from
`config.yaml`: any numeric parameter within 5% of a bound — measured on the
searched scale, so `log: true` ranges are judged in log space — is flagged
with the bound to widen. Categorical parameters have no edges and are skipped.

If something is flagged: widen that bound in `config.yaml`, re-run §6 (the
same study name resumes, and the sampler explores the new region; rename the
study to restart clean), then re-read §7–§8 until the front sits inside the
box. `tsim optimize` prints the same check at the end of every CLI run.

In [ ]:
report = opt.boundary_report(best.params, strat.optuna["space"], front=front)
flagged = report[report["action"] != "ok"]
print("all clear — the searched ranges bracket the optimum" if flagged.empty
      else f"{len(flagged)} parameter(s) pressing against the search space:")
report

## 9 · Re-run the full backtest with the picked parameters

In [ ]:
print("picked params:", best.params)
pf_best = bt.run_strategy(strat, panel, best.params, settings=settings)
stats_best = bt.portfolio_stats(pf_best)
stats_best |= bt.active_stats(pf_best, panel, fees=fees, slippage=slippage,
                              total_return=stats_best["total_return"])
pd.Series(stats_best).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf_best, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity with picked parameters").show()

## 10 · Verification — per-symbol deep dive

Portfolio means can hide an edge carried by a single name, so verify the
picked parameters symbol by symbol. First the full per-symbol breakdown of the
optimized backtest; then set `SYMBOL` in the next cell to any name that looks
interesting (the big winner, the one loser, the one with suspiciously few
trades) and re-run it to get the same annotated chart and trade table as §4 —
entries, exits, direction and the channel each breakout escaped.

In [ ]:
bt.symbol_stats(pf_best).round(4)

In [ ]:
SYMBOL = None   # ← set to any symbol from the table above (e.g. "NVDA"); None = most-traded
sym = SYMBOL or bt.trade_records(pf_best)["symbol"].value_counts().index[0]
inspect_symbol(sym, pf_best, best.params)

In [ ]:
# Gap-fill optimism scan: with eod_exit=false an overnight gap through the stop
# fills AT the stop level in the engine — better than any real fill. Exits on a
# session's FIRST bar are where that optimism lives; if a big share of the P&L
# sits in them, discount the pick accordingly.
tr = bt.trade_records(pf_best)
ts = pd.Series(panel.close.index, index=panel.close.index)
first_bars = set(ts.groupby(panel.close.index.tz_convert(panel.tz).date).min())
open_exits = tr[tr["exit_ts"].isin(first_bars)]
print(f"{len(open_exits)} of {len(tr)} trades exit on a session's first bar; "
      f"their P&L {open_exits['pnl'].sum():+,.0f} vs total {tr['pnl'].sum():+,.0f}")
open_exits.sort_values("pnl").head(8).round(3)

## 11 · Out-of-sample check — the tail the optimizer never saw

Everything above ran on data up to `data.end`, the **train cutoff**. This
section replays the picked parameters on the untouched tail — the one number
in this notebook the optimizer could not have fit.

The mechanics matter: signals are computed on the **full** history (so the
slow regime EMA and the support window are warmed up on the day the holdout
starts) and only then are the panel and the signal frames sliced to the
holdout rows, so every metric below covers exclusively bars after the cutoff.
Same symbols as the train panel — re-resolving coverage over the holdout
would quietly change the universe. The verdict is the **hurdle**: the pick
generalizes only if the holdout `excess_cagr` and `info_ratio` are both
positive — i.e. it beat the equal-weight buy-and-hold of the same names on
data it never saw. Expect worse numbers than §9; that is the point.

In [ ]:
from dataclasses import replace

assert d.get("end"), "set data.end in config.yaml to keep a holdout"
MIN_OOS_TRADES = 50
cutoff = pd.Timestamp(d["end"], tz="UTC")
full = load_panel(engine, panel.symbols, d["bar"], d["start"], None,
                  session=d.get("session", "regular"), exchanges=settings.exchanges)
after = full.close.index > cutoff
assert after.any(), "no bars stored after data.end — backfill the holdout tail first"
fees, slippage = bt.leg_costs(settings.costs)
oos_panel = replace(full, close=full.close[after], open_=full.open_[after],
                    high=full.high[after], low=full.low[after], volume=full.volume[after])

def oos_portfolio(params):
    """Replay one parameter set on the untouched tail (warm-up sees history)."""
    sig = strat.signals(full, **{**strat.params, **params})
    oos_sig = {k: (v[after] if isinstance(v, pd.DataFrame) else v) for k, v in sig.items()}
    return bt.run(oos_panel, oos_sig, fees=fees, slippage=slippage,
                  init_cash=settings.backtest.init_cash)

def oos_stats(params):
    """Holdout stats incl. the benchmark-relative pair, on the holdout slice."""
    pf = oos_portfolio(params)
    st = bt.portfolio_stats(pf)
    return pf, st | bt.active_stats(pf, oos_panel, fees=fees, slippage=slippage,
                                    total_return=st["total_return"])

pf_oos, oos = oos_stats(best.params)
print(f"holdout: {full.close.index[after][0]:%Y-%m-%d} → {full.close.index[-1]:%Y-%m-%d} "
      f"({int(after.sum())} bars, {oos['n_trades']} trades)")
if oos["n_trades"] < MIN_OOS_TRADES:
    print(f"!! only {oos['n_trades']} OOS trades (< {MIN_OOS_TRADES}) — "
          "insufficient sample; do NOT read the metrics below as a verdict")
verdict = "BEATS" if oos["excess_cagr"] > 0 and oos["info_ratio"] > 0 else "does NOT beat"
print(f"hurdle: the pick {verdict} equal-weight buy-and-hold out of sample "
      f"(excess_cagr {oos['excess_cagr']:+.2%}, info_ratio {oos['info_ratio']:+.2f})")
plots.plot_equity(bt.equity_curve(pf_oos, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — OUT-OF-SAMPLE equity, picked parameters").show()

# Directional consistency: replay the two next-best admissible front rows.
# Params come from the FrozenTrials, NOT the frame rows — iterrows() would
# upcast int params (level_window/max_hold/...) to floats and rolling() raises.
by_number = {t.number: t for t in study.best_trials}
table = {"picked": oos}
for trial_no in eligible.sort_values(objs[0], ascending=False)["trial"]:
    if int(trial_no) == best.number or len(table) >= 3:
        continue
    table[f"trial {int(trial_no)}"] = oos_stats(by_number[int(trial_no)].params)[1]
pd.DataFrame({"train (§9)": stats_best,
              **{f"holdout · {k}": v for k, v in table.items()}}).round(4)

## 12 · Cost stress test — does the edge survive realistic friction?

The always-on cost model uses the global `spread_bps: 2.0` — a liquid-large-cap
number that flatters volatile names. This cell re-prices the **same** picked
trades under two harsher regimes: the repo's standard stress (**6 bps spread /
5 bps slippage**, the squeeze_breakout convention) and a volatile-name
calibration (**10 / 3**, the candle_scalp one). The benchmark is re-priced at
the same stressed costs each time, so the comparison stays fair. If
`excess_cagr` and `info_ratio` survive both re-pricings, the edge has a
chance; if they collapse, §9's green number was liquid-cost fiction. (Env
override for a CLI run:
`TRADING_COSTS__SPREAD_BPS=6 TRADING_COSTS__SLIPPAGE_BPS=5`.)

Read this together with the §10 gap scan: `eod_exit: false` picks lean on
engine-optimistic overnight stop fills, so a pick that only wins with default
costs *and* heavy first-bar exits is double-flattered.

In [ ]:
# Re-price the SAME picked trades under stressed friction and compare. active_stats
# recomputes the benchmark at the SAME stressed costs, so the hurdle stays fair.
sig_best = strat.signals(panel, **{**strat.params, **best.params})

def stats_at(pf, f, s):
    st = bt.portfolio_stats(pf)
    return st | bt.active_stats(pf, panel, fees=f, slippage=s,
                                total_return=st["total_return"])

f_def, s_def = bt.leg_costs(settings.costs)
print(f"round-trip cost  default ≈ {2*(f_def+s_def)*1e4:.1f} bps")
cols = {"default costs (§9)": stats_at(pf_best, f_def, s_def)}
for label, spread, slip in (("stressed 6/5 bps", 6.0, 5.0),
                            ("stressed 10/3 bps", 10.0, 3.0)):
    stressed = settings.costs.model_copy(update={"spread_bps": spread,
                                                 "slippage_bps": slip})
    f_str, s_str = bt.leg_costs(stressed)
    print(f"round-trip cost  {label} ≈ {2*(f_str+s_str)*1e4:.1f} bps")
    pf_str = bt.run(panel, sig_best, fees=f_str, slippage=s_str,
                    init_cash=settings.backtest.init_cash)
    cols[label] = stats_at(pf_str, f_str, s_str)
pd.DataFrame(cols).round(4)

## 13 · Interval read-out — do 1, 5 or 15-minute candles work best?

`tf_minutes` turned the candle-interval question into a parameter: the dip
trigger is evaluated on 1/5/15-minute trailing windows derived from the same
1m bars, while execution always stays on the 1-minute grid (so a coarser
trigger pays no extra fill latency). Slice the search results by it. If one
interval dominates the admissible trials *and* the sweep consistently — and
the §11/§12 checks hold for picks at that interval — that is the answer to
"which candle interval makes most sense". An indifferent front is also an
answer: for these names the trigger granularity did not matter.

In [ ]:
# The interval question, answered three ways: all admissible trials, the
# admissible Pareto front, and the hand-picked sweep grid, each sliced by
# tf_minutes. Trials/front are judged on the objectives (info_ratio,
# excess_cagr — every trial carries them); the sweep on excess_cagr/cagr
# (it keeps no per-combo equity curve, so no info_ratio).
adm = everything[everything["n_trades"] >= MIN_TRADES]
print(f"admissible trials by tf_minutes ({len(adm)} of {len(everything)} total):")
display(adm.assign(tf_minutes=adm["tf_minutes"].astype(int))
        .groupby("tf_minutes")[["info_ratio", "excess_cagr", "n_trades"]]
        .agg(["count", "median", "max"]).round(3))
print("admissible Pareto-front trials by tf_minutes:",
      eligible["tf_minutes"].astype(int).value_counts().sort_index().to_dict())
print("\nsweep medians by tf_minutes (default params elsewhere):")
sweep_stats.assign(tf_minutes=sweep_stats["tf_minutes"].astype(int)) \
    .groupby("tf_minutes")[["excess_cagr", "cagr", "pct_profitable", "n_trades"]] \
    .median().round(3)

## Read the result honestly

* §3–§10 are **in-sample**: the optimizer saw all the data they are judged on.
  §11 is the honest number — and even it gets stale after a few re-reads
  (each "check the holdout, tweak, check again" loop quietly fits the tail
  too). For a fresh view, re-run the pick on a *different universe*.
* Prefer combos where **`pct_profitable` is high** — an edge carried by one
  symbol is usually noise.
* **If `entry_mode: touch` with `min_touches: 0` dominates the front**, the
  "tested level" thesis added nothing over generic dip-buying — report that,
  don't ship it dressed up as a support strategy. Likewise, if the `bounce`
  confirmation reliably beats `touch`, that is the adverse-selection
  literature showing up in the data — worth stating explicitly.
* **`eod_exit: false` picks are flattered twice**: overnight gaps through a
  stop fill AT the stop level (§10's scan quantifies it), and stop legs pay
  no slippage. Treat close-run eod true/false comparisons as biased toward
  `false`.
* More trials on the same data = more chances to overfit. The Pareto front
  going up does not mean the *strategy* got better.

**Add your own strategy:** copy `strategies/_template/`, write `signals()`, fill
in `config.yaml` — this exact notebook flow works unchanged. See the README's
"How to add a strategy".